# Calibrate STCRDab binding probabilities

Fits isotonic calibrators on the immrep23 reference (seen peptides, leakage removed for all three predictors) and applies them to the STCRDab predictions, writing the `binder_prob_calibrated` column of `../data/STCRDab_calibrated/`. Run from `publication/notebooks/` **before** `figure5_benchmark.ipynb`.

Outputs: `../data/STCRDab_calibrated/{gcn,gat,mixtcrpred-pan}_calibrated.tsv` and `../figures/calibration/prob_calibration.{svg,png}`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from scipy.special import logit
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
PUB = Path.cwd().parent
sys.path.insert(0, str(PUB / "lib"))
from benchmarker import Benchmarker
from figure_style import apply_style, save_figure
bmarker = Benchmarker()
flierprops = apply_style()

DATA, RESULTS, FIGURES = PUB / "data", PUB / "results", PUB / "figures"

EPS = 1e-6
def clip(p):
    return np.clip(p, EPS, 1 - EPS)

In [ ]:
# === Calibrators (fit on the reference set) ===
def fit_platt_weighted(p_ref, y_ref, C=1.0):
    X = logit(clip(p_ref)).reshape(-1, 1)
    lr = LogisticRegression(C=C, solver="lbfgs", class_weight="balanced", max_iter=1000)
    lr.fit(X, y_ref)
    return lr

def apply_platt(lr, p):
    return lr.predict_proba(logit(clip(p)).reshape(-1, 1))[:, 1]

def fit_isotonic_weighted(p_ref, y_ref):
    # Class-balanced weights so positives and negatives contribute equally.
    vals, counts = np.unique(y_ref, return_counts=True)
    inv = {c: 1.0 / counts[i] for i, c in enumerate(vals)}
    sample_weight = np.array([inv[int(y)] for y in y_ref])
    ir = IsotonicRegression(out_of_bounds="clip")
    ir.fit(clip(p_ref), y_ref, sample_weight=sample_weight)
    return ir

def calibrate_models(models_ref, y_ref, method="isotonic"):
    out = {}
    for name, p_ref in models_ref.items():
        out[name] = fit_platt_weighted(p_ref, y_ref) if method == "platt" else fit_isotonic_weighted(p_ref, y_ref)
    return out

def apply_calibration(calibrators, models_test, method="isotonic"):
    out = {}
    for name, p_test in models_test.items():
        cal = calibrators[name]
        out[name] = apply_platt(cal, p_test) if method == "platt" else cal.predict(clip(p_test))
    return out

# === Calibration quality metrics ===
def brier_weighted(probs, labels):
    vals, counts = np.unique(labels, return_counts=True)
    inv = {c: 1.0 / counts[i] for i, c in enumerate(vals)}
    w = np.array([inv[int(y)] for y in labels])
    return np.average((probs - labels) ** 2, weights=w)

def ece_classbalanced(probs, labels, n_bins=15):
    def _ece(p, y):
        order = np.argsort(p); p, y = p[order], y[order]
        n = len(p); bin_size = max(1, n // n_bins); ece = 0.0
        for i in range(n_bins):
            start = i * bin_size
            if start >= n:
                break
            end = min(n, (i + 1) * bin_size)
            p_bin, y_bin = p[start:end], y[start:end]
            if len(p_bin) == 0:
                continue
            ece += (len(p_bin) / n) * abs(p_bin.mean() - y_bin.mean())
        return ece
    return 0.5 * (_ece(probs[labels == 1], labels[labels == 1]) +
                  _ece(probs[labels == 0], labels[labels == 0]))

In [ ]:
# === Reference set: immrep23, seen peptides, leakage removed for all three predictors ===
# gcn/gat binder probabilities come from the analyzer binding_score columns; mixtcrpred-pan from its prediction table. Labels = binder.
an = bmarker.read_table(str(DATA / "analyzer" / "immrep23_analyzer_table.tsv")).drop_duplicates("identifier")
mp = (bmarker.read_table(str(DATA / "mixtcrpred_pan" / "immrep23_mixtcrpred-pan_predicted.tsv"))
      [["identifier", "binder_prob"]].drop_duplicates("identifier")
      .rename(columns={"binder_prob": "binder_prob_mpred"}))
an = an.merge(mp, on="identifier", how="left")

mask = ((an["sample_in_train_t2pmhc-gcn"] == False) & (an["sample_in_train_t2pmhc-gat"] == False)
        & (an["sample_in_train_mixtcrpred"] == False)
        & (an["seen_in_t2pmhc-gcn"] == True) & (an["seen_in_t2pmhc-gat"] == True)
        & (an["seen_in_mixtcrpred"] == True)
        & an["binding_score_t2pmhc-gcn"].notna() & an["binding_score_t2pmhc-gat"].notna()
        & an["binder_prob_mpred"].notna())
ref = an[mask].reset_index(drop=True)

y_ref = np.array(ref["binder"])
ref_models = {
    "gcn":       np.array(ref["binding_score_t2pmhc-gcn"]),
    "gat":       np.array(ref["binding_score_t2pmhc-gat"]),
    "mpred_pan": np.array(ref["binder_prob_mpred"]),
}
print(f"reference: {len(ref)} samples, {int(y_ref.sum())} positive, {int((y_ref==0).sum())} negative")

# === Test set: STCRDab. Raw model output is the binder_prob column of the calibrated tables. ===
STC = {"gcn": "gcn", "gat": "gat", "mpred_pan": "mixtcrpred-pan"}
stcrdab = {n: bmarker.read_table(str(DATA / "stcrdab_calibrated" / f"{f}_calibrated.tsv")) for n, f in STC.items()}
test_models = {n: np.array(stcrdab[n]["binder_prob"]) for n in STC}
print("STCRDab sizes:", {n: len(stcrdab[n]) for n in STC})

In [ ]:
# === Fit on the reference, apply to STCRDab ===
calibrators = calibrate_models(ref_models, y_ref, method="isotonic")
calibrated  = apply_calibration(calibrators, test_models, method="isotonic")

# Calibration quality on the reference set.
for name, p_ref in ref_models.items():
    p_cal = apply_calibration(calibrators, {name: p_ref}, "isotonic")[name]
    print(f"{name}: ECE={ece_classbalanced(p_cal, y_ref):.4f}, Brier={brier_weighted(p_cal, y_ref):.4f}")

In [ ]:
# === Supplementary calibration figure ===
#   (a) reference: calibrated probability, negatives vs positives, per model
#   (b) STCRDab positives: raw (outline) vs calibrated (filled), per model
MODEL_ORDER = ["gcn", "gat", "mpred_pan"]
DISPLAY_NAMES = {"gcn": "t2pmhc-GCN", "gat": "t2pmhc-GAT", "mpred_pan": "MixTCRpred-pan"}
COLORS = {"gcn": "#D4815B", "gat": "#C46A4B", "mpred_pan": "#A0A0A0"}

fig = plt.figure(figsize=(7.2, 7))
gs = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.3)
ax_a = [fig.add_subplot(gs[0, i]) for i in range(3)]
ax_b = [fig.add_subplot(gs[1, i]) for i in range(3)]

cal_ref = apply_calibration(calibrators, ref_models, "isotonic")
pos_mask, neg_mask = y_ref == 1, y_ref == 0
for i, name in enumerate(MODEL_ORDER):
    ax = ax_a[i]; color = COLORS[name]; p_cal = cal_ref[name]
    data = [p_cal[neg_mask], p_cal[pos_mask]]
    vp = ax.violinplot(data, positions=[0, 1], widths=0.7, showmeans=False, showmedians=False, showextrema=False)
    for j, body in enumerate(vp["bodies"]):
        body.set_facecolor(color); body.set_edgecolor("black"); body.set_linewidth(0.6); body.set_alpha([0.35, 0.8][j])
    ax.boxplot(data, positions=[0, 1], widths=0.15, patch_artist=True, showfliers=False,
               medianprops=dict(color="white", lw=1.2), boxprops=dict(facecolor="0.25", edgecolor="0.25", lw=0.6),
               whiskerprops=dict(color="0.25", lw=0.6), capprops=dict(color="0.25", lw=0.6))
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Negatives", "Positives"], fontsize=8)
    ax.set_title(DISPLAY_NAMES[name], fontsize=9); ax.set_ylim(-0.02, 1.05)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.set_ylabel("Calibrated Binding Probability" if i == 0 else "");
    if i: ax.set_yticklabels([])

cal_test = apply_calibration(calibrators, test_models, "isotonic")
for i, name in enumerate(MODEL_ORDER):
    ax = ax_b[i]; color = COLORS[name]
    vp_raw = ax.violinplot([test_models[name]], positions=[0], widths=0.7, showmeans=False, showmedians=False, showextrema=False)
    for body in vp_raw["bodies"]:
        body.set_facecolor("none"); body.set_edgecolor(color); body.set_linewidth(1.3); body.set_alpha(0.9)
    vp_cal = ax.violinplot([cal_test[name]], positions=[1], widths=0.7, showmeans=False, showmedians=False, showextrema=False)
    for body in vp_cal["bodies"]:
        body.set_facecolor(color); body.set_edgecolor("black"); body.set_linewidth(0.6); body.set_alpha(0.75)
    for pos, d in [(0, test_models[name]), (1, cal_test[name])]:
        ax.boxplot([d], positions=[pos], widths=0.15, patch_artist=True, showfliers=False,
                   medianprops=dict(color="white", lw=1.2), boxprops=dict(facecolor="0.25", edgecolor="0.25", lw=0.6),
                   whiskerprops=dict(color="0.25", lw=0.6), capprops=dict(color="0.25", lw=0.6))
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Raw", "Calibrated"], fontsize=8)
    ax.set_title(DISPLAY_NAMES[name], fontsize=9); ax.set_ylim(-0.02, 1.05)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.set_ylabel("Binding Probability" if i == 0 else "")
    if i: ax.set_yticklabels([])

ax_a[0].text(-0.3, 1.15, "a", transform=ax_a[0].transAxes, fontsize=14, fontweight="bold", va="top")
ax_b[0].text(-0.3, 1.15, "b", transform=ax_b[0].transAxes, fontsize=14, fontweight="bold", va="top")
fig.text(0.5, 0.97, "Calibrated predictions (IMMREP 2023)", ha="center", fontsize=10, fontweight="bold")
fig.text(0.5, 0.48, "Independent positives (STCRDab)", ha="center", fontsize=10, fontweight="bold")
save_figure(fig, "prob_calibration", subdir="calibration")
plt.show()

In [ ]:
# === Write calibrated probabilities back to the STCRDab tables ===
for n, f in STC.items():
    stcrdab[n]["binder_prob_calibrated"] = calibrated[n]
    bmarker.save_table(stcrdab[n], str(DATA / "stcrdab_calibrated" / f"{f}_calibrated.tsv"))
print("wrote:", [f"{f}_calibrated.tsv" for f in STC.values()])